# 03 - Packet Generation

Test `Packet`, `Gateway`, and node send flow.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.config.settings import Settings
from app.simulation.energy_model import EnergyModel
from app.simulation.gateway import Gateway
from app.simulation.node import Node

In [ ]:
settings = Settings()
energy_model = EnergyModel(settings)

gateway = Gateway(settings.GATEWAY_X, settings.GATEWAY_Y)
node = Node(
    node_id=1,
    x=10.0,
    y=20.0,
    initial_energy=settings.INITIAL_ENERGY,
    energy_model=energy_model,
)

print(gateway)
print(node)

In [ ]:
# Sense -> create packet -> send to gateway
for step in range(5):
    if not node.sense():
        print("Sense failed at step", step)
        break

    packet = node.create_packet(
        step=step,
        size=settings.PACKET_SIZE,
        payload={"temperature": 20.0 + step},
    )
    ok = node.send_packet(gateway, packet)
    print(f"step={step} ok={ok} packet={packet} node={node}")

print("Gateway received:", gateway.total_received())
print(gateway.received_packets)

In [ ]:
# Sleeping node cannot send
node.sleep()
packet = node.create_packet(step=99, size=settings.PACKET_SIZE)
print("Packet while sleeping:", packet)

node.wake_up()
packet = node.create_packet(step=100, size=settings.PACKET_SIZE)
print("Packet after wake:", packet)
print("Send after wake:", node.send_packet(gateway, packet))